In [1]:
"""
=========================================================
07_final_model.py
=========================================================

Purpose
-------
Train the final production CatBoost model using the
final selected features after feature ablation.

Inputs
------
data/processed/feature_engineered.csv
outputs/best_hyperparameters.json

Outputs
-------
models/final_catboost_model.cbm
outputs/final_feature_importance.csv
outputs/final_metrics.json
"""

'\n=========================================================\n07_final_model.py\n=========================================================\n\nPurpose\n-------\nTrain the final production CatBoost model using the\nfinal selected features after feature ablation.\n\nInputs\n------\ndata/processed/feature_engineered.csv\noutputs/best_hyperparameters.json\n\nOutputs\n-------\nmodels/final_catboost_model.cbm\noutputs/final_feature_importance.csv\noutputs/final_metrics.json\n'

In [2]:
import json
import warnings

import pandas as pd

from catboost import CatBoostClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from sklearn.model_selection import train_test_split
from pathlib import Path
warnings.filterwarnings("ignore")

In [3]:
#Global declare random state
RANDOM_STATE = 42

In [ ]:
# ==========================================================
# Paths
# ==========================================================

BASE_DIR = Path.cwd().parent

In [ ]:
##########################################################
# LOAD DATA
##########################################################

print("=" * 70)
print("FINAL MODEL TRAINING")
print("=" * 70)

df = pd.read_csv(f"{BASE_DIR}/data/processed/feature_engineered.csv")

TARGET = "target"

##########################################################
# FINAL FEATURES
##########################################################

FINAL_FEATURES = [

    "current_age",

    "vintage_months",

    "DP_Value_log",

    "Total_Margin_log",

    "ord_cash_log",

    "Category_Bucket_final",

    "self_dealer_status",

    "Trading_member_code",

    "plan",

    "recency",

    "never_traded"

]

In [ ]:
##########################################################
# CATEGORICAL FEATURES
##########################################################

categorical_columns = [

    "Category_Bucket_final",

    "self_dealer_status",

    "Trading_member_code",

    "plan"

]

In [ ]:
##########################################################
# DATA PREPARATION
##########################################################

X = df[FINAL_FEATURES].copy()
y = df[TARGET]

for col in categorical_columns:

    X[col] = X[col].fillna("Missing").astype(str)

In [ ]:
##########################################################
# TRAIN / VALID / TEST
##########################################################

X_train, X_temp, y_train, y_temp = train_test_split(

    X,
    y,

    test_size=0.30,

    stratify=y,

    random_state=RANDOM_STATE

)

X_valid, X_test, y_valid, y_test = train_test_split(

    X_temp,
    y_temp,

    test_size=0.50,

    stratify=y_temp,

    random_state=RANDOM_STATE

)

In [ ]:
##########################################################
# LOAD BEST PARAMETERS
##########################################################

with open("outputs/best_hyperparameters.json") as f:

    BEST_PARAMS = json.load(f)

BEST_PARAMS.update({

    "loss_function": "Logloss",

    "eval_metric": "AUC",

    "random_seed": RANDOM_STATE,

    "verbose": 100

})

In [ ]:
##########################################################
# TRAIN MODEL
##########################################################

cat_features = [

    X.columns.get_loc(col)

    for col in categorical_columns

]

model = CatBoostClassifier(**BEST_PARAMS)

model.fit(

    X_train,
    y_train,

    cat_features=cat_features,

    eval_set=(X_valid, y_valid),

    use_best_model=True

)

In [ ]:
##########################################################
# EVALUATION FUNCTION
##########################################################

def evaluate(name, X_data, y_true):

    pred_prob = model.predict_proba(X_data)[:,1]

    pred = model.predict(X_data)

    roc = roc_auc_score(y_true, pred_prob)

    pr = average_precision_score(y_true, pred_prob)

    gini = 2 * roc - 1

    print("\n" + "="*60)
    print(name.upper())
    print("="*60)

    print(f"ROC AUC : {roc:.6f}")
    print(f"PR AUC  : {pr:.6f}")
    print(f"Gini    : {gini:.6f}")

    print("\nConfusion Matrix")

    print(confusion_matrix(y_true, pred))

    print("\nClassification Report")

    print(classification_report(y_true, pred))

    return roc, pr, gini

In [ ]:
##########################################################
# RESULTS
##########################################################

train_metrics = evaluate(

    "Train",

    X_train,

    y_train

)

validation_metrics = evaluate(

    "Validation",

    X_valid,

    y_valid

)

test_metrics = evaluate(

    "Test",

    X_test,

    y_test

)

In [ ]:
##########################################################
# FEATURE IMPORTANCE
##########################################################

importance = pd.DataFrame({

    "Feature": FINAL_FEATURES,

    "Importance": model.get_feature_importance()

})

importance = importance.sort_values(

    "Importance",

    ascending=False

)

print("\n")

print("="*60)
print("FEATURE IMPORTANCE")
print("="*60)

print(importance)

importance.to_csv(

    f"{BASE_DIR}/outputs/final_feature_importance.csv",

    index=False

)

In [ ]:
##########################################################
# SAVE METRICS
##########################################################

metrics = {

    "Train":{

        "ROC_AUC":train_metrics[0],

        "PR_AUC":train_metrics[1],

        "GINI":train_metrics[2]

    },

    "Validation":{

        "ROC_AUC":validation_metrics[0],

        "PR_AUC":validation_metrics[1],

        "GINI":validation_metrics[2]

    },

    "Test":{

        "ROC_AUC":test_metrics[0],

        "PR_AUC":test_metrics[1],

        "GINI":test_metrics[2]

    }

}

with open(

    f"{BASE_DIR}/outputs/final_metrics.json",

    "w"

) as f:

    json.dump(

        metrics,

        f,

        indent=4

    )

In [ ]:
##########################################################
# SAVE MODEL
##########################################################

model.save_model(

    f"{BASE_DIR}/models/final_catboost_model.cbm"

)

print("\n")

print("="*60)

print("MODEL SAVED")

print("="*60)

print(f"{BASE_DIR}/models/final_catboost_model.cbm")

print(f"{BASE_DIR}/outputs/final_feature_importance.csv")

print(f"{BASE_DIR}/outputs/final_metrics.json")